# Notebook 8: Uplift Modeling - Heterogeneous Treatment Effects

**Finding Who Benefits Most from the Email Campaign**

## Overview

In typical A/B testing, we calculate the **average** effect of the email: "On average, customers who receive Mens Email have X% higher conversion rate." But here's the catch: not everyone responds the same way!

**Uplift modeling** asks: "For which individual customers is the email beneficial?" Some customers might be more likely to buy anyway (with or without email), while others might be genuinely persuaded by it. This notebook demonstrates causal machine learning techniques to identify customers with the highest individual treatment effects.

**New in this version:** stratified uplift analysis by `buyer_type` to disentangle customer-quality effects from incremental treatment effects (resolves the nb01 Mixed-group confound).

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb08', exist_ok=True)


In [2]:
# Load data
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment groups:")
print(df['segment'].value_counts())

# For this analysis, focus on Mens Email vs No Email
df_analysis = df[df['segment'].isin(['Mens E-Mail', 'No E-Mail'])].copy()
df_analysis['treatment'] = (df_analysis['segment'] == 'Mens E-Mail').astype(int)

print(f"\nAnalysis sample size: {len(df_analysis)}")
print(f"Conversion rate by group:")
print(df_analysis.groupby('segment')['conversion'].mean())

Dataset shape: (64000, 24)

Treatment groups:
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Analysis sample size: 42613
Conversion rate by group:
segment
Mens E-Mail    0.012531
No E-Mail      0.005726
Name: conversion, dtype: float64


## Concept: Average Treatment Effect vs Heterogeneous Treatment Effects

### Average Treatment Effect (ATE)
The average difference between treatment and control groups across all customers.
- Easy to calculate
- Good for policy decisions ("Should we send emails to everyone?")
- But misses variation within groups

### Heterogeneous Treatment Effects (HTE) / Uplift
The individual treatment effect for each customer.
- Depends on customer characteristics
- Useful for targeted decisions ("Who should we email?")
- Requires more sophisticated modeling

### Four Customer Types (Qini Classification)

```
                    | Receives Email | No Email
Buys Anyway         |   ✓ (Spend)   | ✓(Lost cause)
Won't Buy           |   ✗ (Wasted)  | ✗
Persuadable/Uplift  |   ✓ (Benefit) | ✗ (Your targets!)
Sleeping Dogs       |   ? (Backlash)| Could be fine
```

**Our Goal:** Identify "Persuadables" - customers more likely to convert BECAUSE of the email.

### Why This Matters

1. **Targeting**: Send emails only to high-uplift customers
2. **Cost Reduction**: Avoid wasting emails on lost causes
3. **Efficiency**: Focus budget on customers who respond
4. **Personalization**: Different messaging for different types

## Why Uplift Modeling Resolves the "Mixed-Group" Confound

In nb01 we observed that customers who historically buy **both** men's and women's products (the `Mixed` / cross-shopper segment) have a much higher raw visit rate (28.4%) and conversion rate (1.95%) after receiving any email than the `Match` (16.9% / 1.09%) or `Mismatch` (13.9% / 0.85%) groups.

That raw comparison is confounded: cross-shoppers are inherently more engaged customers, so they have **higher baseline outcomes** regardless of treatment. The naive email-match comparison mixes two effects:

1. **Customer quality** — cross-shoppers would buy more even without any email
2. **Treatment effect** — the email's actual incremental impact

Uplift modeling untangles these by estimating, for each customer:

$$\tau(x) = E[Y \mid X=x, W=1] - E[Y \mid X=x, W=0]$$

Because `mens`, `womens`, and `cross_shopper` are all features in the model, a cross-shopper is compared against **their own conditional baseline**, not against the overall population. The high baseline propensity of cross-shoppers gets absorbed into $E[Y \mid X, W=0]$ and subtracted out — leaving only the true incremental lift attributable to the email.

Below we (a) include `cross_shopper` explicitly as a feature so its role is interpretable in feature importance, and (b) report uplift stratified by `buyer_type` so we can see whether the cross-shopper segment's *incremental* response is actually high, or whether nb01's raw signal was entirely baseline-driven.

## Two-Model Approach (T-Learner)

**T-Learner** trains separate models for treatment and control groups.

### Algorithm
1. **Model 1 (Control)**: Train on customers who didn't receive email
   - Learns: P(conversion | features, no email)
2. **Model 2 (Treatment)**: Train on customers who received email
   - Learns: P(conversion | features, email)
3. **Uplift**: For each customer, calculate
   - Uplift = P(conversion | features, email) - P(conversion | features, no email)
4. **Ranking**: Rank customers by uplift score

### Advantages
- Straightforward interpretation
- Can use different model types for each group
- Works well when treatment effect varies with features

In [3]:
# Prepare features for uplift modeling
# Encode categorical variables
df_model = df_analysis.copy()

# Encode channel (if categorical)
if df_model['channel'].dtype == 'object':
    le_channel = LabelEncoder()
    df_model['channel_encoded'] = le_channel.fit_transform(df_model['channel'])
else:
    df_model['channel_encoded'] = df_model['channel']

# Encode zip_code if categorical (just use numeric codes)
if df_model['zip_code'].dtype == 'object':
    le_zip = LabelEncoder()
    df_model['zip_encoded'] = le_zip.fit_transform(df_model['zip_code'])
else:
    df_model['zip_encoded'] = df_model['zip_code']

# Ensure cross_shopper exists (engineered in nb01; recompute as fallback)
if 'cross_shopper' not in df_model.columns:
    df_model['cross_shopper'] = ((df_model['mens'] == 1) & (df_model['womens'] == 1)).astype(int)

# Select features for modeling.
# NOTE: `cross_shopper` added explicitly so feature-importance directly
# answers "does cross-shopper status drive differential treatment response?"
feature_cols = ['recency', 'history', 'mens', 'womens', 'cross_shopper',
                'zip_encoded', 'newbie', 'channel_encoded']
X = df_model[feature_cols]
y = df_model['conversion'].astype(int)
W = df_model['treatment']

print(f"Features: {feature_cols}")
print(f"Sample size: {len(X)}")
print(f"Treatment rate: {W.mean():.2%}")
print(f"Conversion rate: {y.mean():.2%}")
print(f"Cross-shopper rate: {df_model['cross_shopper'].mean():.2%}")
print(f"\nFeature statistics:")
print(X.describe())

Features: ['recency', 'history', 'mens', 'womens', 'cross_shopper', 'zip_encoded', 'newbie', 'channel_encoded']
Sample size: 42613
Treatment rate: 50.00%
Conversion rate: 0.91%
Cross-shopper rate: 10.16%

Feature statistics:
            recency       history          mens        womens  cross_shopper  \
count  42613.000000  42613.000000  42613.000000  42613.000000   42613.000000   
mean       5.761669    241.859315      0.552085      0.549527       0.101612   
std        3.505422    256.574723      0.497286      0.497547       0.302141   
min        1.000000     29.990000      0.000000      0.000000       0.000000   
25%        2.000000     64.500000      0.000000      0.000000       0.000000   
50%        6.000000    157.000000      1.000000      1.000000       0.000000   
75%        9.000000    325.210000      1.000000      1.000000       0.000000   
max       12.000000   3345.930000      1.000000      1.000000       1.000000   

        zip_encoded        newbie  channel_encoded  
c

In [4]:
# T-Learner: Train separate models
# Split data by treatment status
X_control = X[W == 0]
y_control = y[W == 0]
X_treatment = X[W == 1]
y_treatment = y[W == 1]

print(f"Control sample: {len(X_control)}")
print(f"Treatment sample: {len(X_treatment)}")
print(f"Control conversion: {y_control.mean():.2%}")
print(f"Treatment conversion: {y_treatment.mean():.2%}")

# Train logistic regression models (simpler, interpretable)
model_control = LogisticRegression(max_iter=1000, random_state=42)
model_control.fit(X_control, y_control)

model_treatment = LogisticRegression(max_iter=1000, random_state=42)
model_treatment.fit(X_treatment, y_treatment)

# Predict for all customers
pred_prob_control = model_control.predict_proba(X)[:, 1]  # P(Y=1 | X, no email)
pred_prob_treatment = model_treatment.predict_proba(X)[:, 1]  # P(Y=1 | X, email)

# Calculate uplift
df_model['uplift_tlearner'] = pred_prob_treatment - pred_prob_control

print(f"\n=== T-Learner Uplift Results ===")
print(f"Mean predicted control conversion: {pred_prob_control.mean():.4f}")
print(f"Mean predicted treatment conversion: {pred_prob_treatment.mean():.4f}")
print(f"Average uplift: {df_model['uplift_tlearner'].mean():.4f}")
print(f"\nUplift distribution:")
print(df_model['uplift_tlearner'].describe())
print(f"\nPositive uplift (good targets): {(df_model['uplift_tlearner'] > 0).mean():.1%}")

Control sample: 21306
Treatment sample: 21307
Control conversion: 0.57%
Treatment conversion: 1.25%

=== T-Learner Uplift Results ===
Mean predicted control conversion: 0.0058
Mean predicted treatment conversion: 0.0125
Average uplift: 0.0068

Uplift distribution:
count    42613.000000
mean         0.006777
std          0.003173
min          0.000825
25%          0.005366
50%          0.005895
75%          0.006634
max          0.052301
Name: uplift_tlearner, dtype: float64

Positive uplift (good targets): 100.0%


## Single-Model Approach (S-Learner)

**S-Learner** trains one model with treatment as a feature.

### Algorithm
1. Train one model: P(Y | X, Treatment)
2. For each customer, predict twice:
   - P(Y | X, Treatment=1) with email
   - P(Y | X, Treatment=0) without email
3. Uplift = Difference

### Advantages
- Simpler training
- Regularization helps with variance
- Good baseline

### Disadvantages  
- May struggle to capture interaction effects
- Treatment coefficient alone doesn't give individual effects

In [5]:
# S-Learner: Single model with treatment as feature
X_with_treatment = X.copy()
X_with_treatment['treatment'] = W.values

model_s = LogisticRegression(max_iter=1000, random_state=42)
model_s.fit(X_with_treatment, y)

# Predict with treatment=1 and treatment=0
X_treated = X_with_treatment.copy()
X_treated['treatment'] = 1
X_control_s = X_with_treatment.copy()
X_control_s['treatment'] = 0

pred_treated_s = model_s.predict_proba(X_treated)[:, 1]
pred_control_s = model_s.predict_proba(X_control_s)[:, 1]

df_model['uplift_slearner'] = pred_treated_s - pred_control_s

print("=== S-Learner Uplift Results ===")
print(f"Average uplift: {df_model['uplift_slearner'].mean():.4f}")
print(f"Positive uplift: {(df_model['uplift_slearner'] > 0).mean():.1%}")

# Compare T-Learner vs S-Learner
correlation = np.corrcoef(df_model['uplift_tlearner'], df_model['uplift_slearner'])[0, 1]
print(f"\nCorrelation between T-Learner and S-Learner: {correlation:.4f}")
print(f"\nBoth methods identify similar customers, but with different magnitudes."
      f"\nT-Learner typically more flexible for heterogeneous effects.")

=== S-Learner Uplift Results ===
Average uplift: 0.0068
Positive uplift: 100.0%

Correlation between T-Learner and S-Learner: 0.5747

Both methods identify similar customers, but with different magnitudes.
T-Learner typically more flexible for heterogeneous effects.


## Class Variable Transformation (Jaskowski Approach)

An alternative approach for binary outcomes:

1. Create transformed outcome:
   - Z = Y * W / P(W=1) - Y * (1-W) / P(W=0)
   - Where W is treatment indicator
2. Train one model predicting Z from features
3. Predicted Z is the uplift

**Intuition:** This transformation weights the outcomes by treatment probability, making the model focus on treatment response.

In [6]:
# Class Variable Transformation approach
p_treatment = W.mean()

# Create transformed outcome Z
Z = (y.values * W.values / p_treatment - 
     y.values * (1 - W.values) / (1 - p_treatment))

# Train model on Z
model_z = LinearRegression()
model_z.fit(X, Z)

df_model['uplift_transformed'] = model_z.predict(X)

print("=== Class Variable Transformation Results ===")
print(f"Mean transformed outcome: {Z.mean():.4f}")
print(f"Average predicted uplift: {df_model['uplift_transformed'].mean():.4f}")

# Compare all three methods
print(f"\n=== Comparison of All Methods ===")
methods_corr = pd.DataFrame({
    'T-Learner': df_model['uplift_tlearner'],
    'S-Learner': df_model['uplift_slearner'],
    'Transformed': df_model['uplift_transformed']
}).corr()

print(methods_corr)

=== Class Variable Transformation Results ===
Mean transformed outcome: 0.0068
Average predicted uplift: 0.0068

=== Comparison of All Methods ===
             T-Learner  S-Learner  Transformed
T-Learner     1.000000   0.574710     0.949562
S-Learner     0.574710   1.000000     0.597004
Transformed   0.949562   0.597004     1.000000


In [7]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Uplift Evaluation: Qini Curve and AUUC

### Qini Curve
Shows cumulative incremental conversions as we target higher-uplift customers.

**x-axis:** % of population targeted (ordered by uplift, highest first)
**y-axis:** Cumulative incremental conversions vs random targeting

### Area Under Uplift Curve (AUUC)
Summary metric: area between Qini curve and random baseline.
- Higher AUUC = better model
- Can compare different uplift models

### Qini Coefficient
Normalized version of AUUC (0 to 1 scale)

In [8]:
def calculate_qini_curve(df, uplift_col, y_col, treatment_col):
    """Calculate Qini curve for uplift model."""
    # Sort by predicted uplift (descending)
    df_sorted = df.sort_values(uplift_col, ascending=False).reset_index(drop=True)
    
    n = len(df_sorted)
    n_treatment = df_sorted[treatment_col].sum()
    n_control = n - n_treatment
    
    # For each percentile, calculate incremental lift
    percentiles = np.arange(0, n+1)
    qini = np.zeros(len(percentiles))
    
    for i, pct in enumerate(percentiles):
        if pct == 0:
            qini[i] = 0
        else:
            subset = df_sorted.iloc[:int(pct*n/100)]
            treatment_conversions = subset[subset[treatment_col] == 1][y_col].sum()
            control_conversions = subset[subset[treatment_col] == 0][y_col].sum()
            
            # Normalize by group sizes in sample
            if subset[treatment_col].sum() > 0:
                treatment_rate = treatment_conversions / subset[treatment_col].sum()
            else:
                treatment_rate = 0
            
            if (subset[treatment_col] == 0).sum() > 0:
                control_rate = control_conversions / (subset[treatment_col] == 0).sum()
            else:
                control_rate = 0
            
            # Incremental = (treatment_rate - control_rate) * n_in_sample
            qini[i] = (treatment_rate - control_rate) * len(subset)
    
    return percentiles, qini

# Calculate Qini curves for all methods
percentiles_t, qini_t = calculate_qini_curve(df_model, 'uplift_tlearner', 'conversion', 'treatment')
percentiles_s, qini_s = calculate_qini_curve(df_model, 'uplift_slearner', 'conversion', 'treatment')
percentiles_z, qini_z = calculate_qini_curve(df_model, 'uplift_transformed', 'conversion', 'treatment')

# Random baseline (target by random order)
qini_random = percentiles_t * (df_model['conversion'].mean())

print("Qini Curve calculated for all three methods")

Qini Curve calculated for all three methods


In [9]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Targeting Strategy

Once we have uplift predictions, we can develop a targeting strategy:

1. **Rank all customers by predicted uplift**
2. **Target top percentiles** where uplift is highest
3. **Calculate expected lift** vs random or universal sending
4. **Identify optimal targeting depth** (sweet spot in Qini curve)

The Qini curve tells us: "If we target the top X% of customers by uplift, how many incremental conversions do we gain?"

**Business Decision:** How much margin do we need per conversion to make emailing worthwhile?

In [10]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

In [11]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Stratified Uplift by Buyer Type — Resolving the Mixed-Group Signal

The table and chart below compare, for each `buyer_type` segment:
- **Baseline conversion** — the rate for untreated customers in that segment
- **Treated conversion** — the rate for treated customers in that segment
- **Predicted uplift** — the model's estimate of incremental lift from receiving an email

**Read this table as:** if cross-shoppers (`both`) have a much higher baseline than single-category buyers but a *similar or smaller* predicted uplift, then nb01's "Mixed group outperforms" finding was primarily customer-quality driven, not a differential treatment effect. If instead cross-shoppers also show a much higher uplift, then the email genuinely works better on them and they're a valid targeting segment.

In [12]:
# Stratified uplift by buyer_type — disentangles customer quality from treatment effect
# buyer_type was engineered in nb01: 'mens_only', 'womens_only', 'both' (cross-shopper)
if 'buyer_type' not in df_model.columns:
    def _bt(row):
        if row['mens'] == 1 and row['womens'] == 0: return 'mens_only'
        if row['mens'] == 0 and row['womens'] == 1: return 'womens_only'
        return 'both'
    df_model['buyer_type'] = df_model.apply(_bt, axis=1)

strat = (df_model.groupby('buyer_type')
         .apply(lambda g: pd.Series({
             'n': len(g),
             'baseline_conv': g.loc[g['treatment']==0, 'conversion'].mean(),
             'treated_conv':  g.loc[g['treatment']==1, 'conversion'].mean(),
             'observed_lift': (g.loc[g['treatment']==1, 'conversion'].mean()
                               - g.loc[g['treatment']==0, 'conversion'].mean()),
             'predicted_uplift': g['uplift_tlearner'].mean(),
         }), include_groups=False)
         .reindex(['mens_only','womens_only','both']))

strat_pretty = strat.copy()
strat_pretty['n'] = strat_pretty['n'].astype(int)
for c in ['baseline_conv','treated_conv','observed_lift','predicted_uplift']:
    strat_pretty[c] = (strat_pretty[c]*100).round(3).astype(str) + '%'

print("=== Uplift Stratified by Buyer Type ===")
print(strat_pretty.to_string())

# Visualize: baseline vs treated vs predicted-uplift by buyer_type

# Save stratified results for downstream reference / blog embed
strat.to_csv('../data/outputs/nb08/nb08_uplift_by_buyer_type.csv')
print("\n✓ Saved: ../data/outputs/nb08/nb08_uplift_by_buyer_type.csv")

# Plain-language interpretation
print("\n=== Interpretation ===")
both_baseline = strat.loc['both','baseline_conv']
both_uplift = strat.loc['both','predicted_uplift']
mens_uplift = strat.loc['mens_only','predicted_uplift']
womens_uplift = strat.loc['womens_only','predicted_uplift']
print(f"Cross-shoppers (both): baseline = {both_baseline:.2%}, predicted uplift = {both_uplift:.2%}")
print(f"Mens-only buyers:      predicted uplift = {mens_uplift:.2%}")
print(f"Womens-only buyers:    predicted uplift = {womens_uplift:.2%}")
print("\nIf cross-shopper uplift is close to or below single-category uplift,")
print("the raw 'Mixed group outperforms' finding from nb01 was primarily baseline")
print("customer-quality, NOT a differential treatment effect.")

=== Uplift Stratified by Buyer Type ===
                 n baseline_conv treated_conv observed_lift predicted_uplift
buyer_type                                                                  
mens_only    19196         0.55%       1.151%        0.601%           0.593%
womens_only  19087        0.515%       1.087%        0.572%           0.577%
both          4330        0.931%        2.43%        1.499%           1.498%

✓ Saved: ../data/outputs/nb08/nb08_uplift_by_buyer_type.csv

=== Interpretation ===
Cross-shoppers (both): baseline = 0.93%, predicted uplift = 1.50%
Mens-only buyers:      predicted uplift = 0.59%
Womens-only buyers:    predicted uplift = 0.58%

If cross-shopper uplift is close to or below single-category uplift,
the raw 'Mixed group outperforms' finding from nb01 was primarily baseline
customer-quality, NOT a differential treatment effect.


In [13]:
# Feature importance: which features drive differential response?
# Use random forest for better feature importance
from sklearn.ensemble import RandomForestClassifier

# Train RF models separately (T-Learner)
rf_control = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_control.fit(X_control, y_control)

rf_treatment = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_treatment.fit(X_treatment, y_treatment)

# Calculate difference in feature importance
importance_control = rf_control.feature_importances_
importance_treatment = rf_treatment.feature_importances_
importance_diff = importance_treatment - importance_control

feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Control Importance': importance_control,
    'Treatment Importance': importance_treatment,
    'Difference': importance_diff
}).sort_values('Difference', ascending=True)

print("\n=== Feature Importance for Uplift ===")
print(feature_importance_df)

# Visualize


=== Feature Importance for Uplift ===
           Feature  Control Importance  Treatment Importance  Difference
1          history            0.801828              0.775393   -0.026435
4    cross_shopper            0.007924              0.006035   -0.001889
7  channel_encoded            0.039352              0.038290   -0.001062
2             mens            0.006575              0.006107   -0.000468
3           womens            0.004822              0.006338    0.001515
6           newbie            0.014738              0.018866    0.004128
5      zip_encoded            0.034994              0.042166    0.007172
0          recency            0.089766              0.106805    0.017039


In [14]:
# --- Compute AUUC, optimal targeting %, and incremental conversions ---
import numpy as np
_u = df_model['uplift_tlearner'].values
_y = df_model['conversion'].values
_w = df_model['treatment'].values
_order = np.argsort(-_u)
_y_s = _y[_order]; _w_s = _w[_order]
_n = len(_y_s)
_cum_t = np.cumsum(_w_s); _cum_c = np.cumsum(1 - _w_s)
_cum_yt = np.cumsum(_y_s * _w_s); _cum_yc = np.cumsum(_y_s * (1 - _w_s))
with np.errstate(invalid='ignore', divide='ignore'):
    _rate_t = np.where(_cum_t > 0, _cum_yt/_cum_t, 0.0)
    _rate_c = np.where(_cum_c > 0, _cum_yc/_cum_c, 0.0)
_qini = (_rate_t - _rate_c) * np.arange(1, _n+1)
_fracs = np.arange(1, _n+1) / _n
_rand = _fracs * _qini[-1]
auuc_t = float(np.trapz(_qini - _rand, _fracs))
# Incremental conversions at each targeting decile (1% to 100%)
_deciles = np.arange(1, 101)
incremental_conversions = np.array([_qini[int(d/100 * _n) - 1] for d in _deciles])
optimal_pct = int(_deciles[int(np.argmax(incremental_conversions))])

# Save key results
uplift_results = df_model[['conversion', 'treatment', 'spend', 'recency', 'history', 
                             'uplift_tlearner', 'uplift_slearner', 'uplift_transformed']].copy()
uplift_results.to_csv('../data/outputs/nb08/nb08_uplift_results.csv', index=False)

# Create summary table
summary_uplift = pd.DataFrame({
    'Metric': [
        'Average Predicted Uplift (T-Learner)',
        'Std Dev Uplift',
        '% Customers with Positive Uplift',
        'Max Predicted Uplift',
        'Min Predicted Uplift',
        'AUUC (T-Learner)',
        'Optimal Targeting Percentage',
        'Expected Incremental Conversions (Optimal)'
    ],
    'Value': [
        f"{df_model['uplift_tlearner'].mean():.4f}",
        f"{df_model['uplift_tlearner'].std():.4f}",
        f"{(df_model['uplift_tlearner'] > 0).mean():.1%}",
        f"{df_model['uplift_tlearner'].max():.4f}",
        f"{df_model['uplift_tlearner'].min():.4f}",
        f"{auuc_t:.4f}",
        f"{optimal_pct}%",
        f"{incremental_conversions[optimal_pct-1]:.0f}"
    ]
})

print("\n=== UPLIFT MODELING SUMMARY ===")
print(summary_uplift.to_string(index=False))

summary_uplift.to_csv('../data/outputs/nb08/nb08_uplift_summary.csv', index=False)
print("\nResults saved to: ../data/outputs/nb08/nb08_uplift_results.csv")
print("Summary saved to: ../data/outputs/nb08/nb08_uplift_summary.csv")


=== UPLIFT MODELING SUMMARY ===
                                    Metric   Value
      Average Predicted Uplift (T-Learner)  0.0068
                            Std Dev Uplift  0.0032
          % Customers with Positive Uplift  100.0%
                      Max Predicted Uplift  0.0523
                      Min Predicted Uplift  0.0008
                          AUUC (T-Learner) 29.0215
              Optimal Targeting Percentage     99%
Expected Incremental Conversions (Optimal)     290

Results saved to: ../data/outputs/nb08/nb08_uplift_results.csv
Summary saved to: ../data/outputs/nb08/nb08_uplift_summary.csv


## Key Takeaways

1. **Not Everyone Responds the Same**: Uplift modeling reveals heterogeneous treatment effects
2. **Three Methods Compared**:
   - T-Learner: Train separate models (most flexible)
   - S-Learner: Single model with treatment feature (simpler)
   - Class Variable Transformation: Specialized for classification
3. **Qini Curve**: Shows how much value we gain by targeting high-uplift customers
4. **Targeting Strategy**: Use uplift scores to identify which customers to email for maximum ROI
5. **Feature Analysis**: Understand which customer characteristics drive differential responses

## When to Use Uplift Modeling

- **Personalized campaigns**: Target only customers likely to respond
- **Cost optimization**: Send emails only when ROI is positive
- **Segmentation**: Create targeted messaging for different customer types
- **Campaign design**: Choose channels/timing based on predicted uplift

## Limitations

- Requires sufficient sample size in both treatment and control
- Can overfit if not careful with feature selection
- Assumes no unmeasured confounders (treatment is truly random)
- Predictions are noisier than ATE estimates

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [15]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb08")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb08


In [17]:
# Chart 1: Stratified uplift by buyer_type (from earlier stratified analysis)
# Reload from saved CSV so this cell works standalone
strat_path = os.path.join(OUT_DIR, "nb08_uplift_by_buyer_type.csv")
if os.path.exists(strat_path):
    strat = pd.read_csv(strat_path, index_col=0)
    strat = strat.reindex(["mens_only","womens_only","both"])
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("Baseline vs Treated Conversion", "Predicted Uplift"),
                        horizontal_spacing=0.14)
    # Grouped bars: baseline vs treated
    fig.add_trace(go.Bar(x=strat.index, y=strat["baseline_conv"]*100, name="Baseline",
                         marker_color="#95A5A6",
                         text=[f"{v:.2%}" for v in strat["baseline_conv"]], textposition="outside",
                         hovertemplate="<b>%{x}</b><br>Baseline: %{y:.3f}%<extra></extra>"),
                  row=1, col=1)
    fig.add_trace(go.Bar(x=strat.index, y=strat["treated_conv"]*100, name="Treated",
                         marker_color="#4C8BB8",
                         text=[f"{v:.2%}" for v in strat["treated_conv"]], textposition="outside",
                         hovertemplate="<b>%{x}</b><br>Treated: %{y:.3f}%<extra></extra>"),
                  row=1, col=1)
    # Predicted uplift
    colors = ["#4C8BB8","#5FA85F","#F39C12"]
    fig.add_trace(go.Bar(x=strat.index, y=strat["predicted_uplift"]*100,
                         marker_color=colors, showlegend=False,
                         text=[f"{v*100:+.2f}pp" for v in strat["predicted_uplift"]],
                         textposition="outside",
                         hovertemplate="<b>%{x}</b><br>Uplift: %{y:.3f} pp<extra></extra>"),
                  row=1, col=2)
    fig.update_xaxes(automargin=True)
    fig.update_yaxes(title_text="Conversion Rate (%)", automargin=True, row=1, col=1)
    fig.update_yaxes(title_text="Predicted Uplift (pp)", automargin=True, row=1, col=2)
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=70, r=40, t=100, b=90)},
                      title="Uplift Stratified by Buyer Type — Resolves the Mixed-Group Confound",
                      height=520, barmode="group")
    fig.write_html(os.path.join(OUT_DIR, "nb08_uplift_by_buyer_type_interactive.html"), **PLOTLY_KW)
    fig.show()
    print("  ✓ nb08_uplift_by_buyer_type_interactive.html")
else:
    print("  ! nb08_uplift_by_buyer_type.csv not found — run earlier cells first")


  ✓ nb08_uplift_by_buyer_type_interactive.html


In [18]:
# Chart 2: Qini curve — rebuild quickly using saved model uplift if available
# We reconstruct from the clean CSV using a simple T-Learner (logistic regression)
from sklearn.linear_model import LogisticRegression
df8 = df_blog.copy()
if df8["zip_code"].dtype == object:
    df8["zip_encoded"] = df8["zip_code"].astype("category").cat.codes
else:
    df8["zip_encoded"] = df8["zip_code"]
if df8["channel"].dtype == object:
    df8["channel_encoded_local"] = df8["channel"].astype("category").cat.codes
else:
    df8["channel_encoded_local"] = df8["channel"]
df8["cross_shopper"] = ((df8["mens"]==1) & (df8["womens"]==1)).astype(int)
fcols = ["recency","history","mens","womens","cross_shopper","zip_encoded","newbie","channel_encoded_local"]
X8 = df8[fcols].values; W8 = df8["treatment"].values; y8 = df8["conversion"].values

# T-Learner
m0 = LogisticRegression(max_iter=200).fit(X8[W8==0], y8[W8==0])
m1 = LogisticRegression(max_iter=200).fit(X8[W8==1], y8[W8==1])
p0 = m0.predict_proba(X8)[:,1]
p1 = m1.predict_proba(X8)[:,1]
uplift = p1 - p0

# Qini curve
ord_idx = np.argsort(-uplift)
y_sorted = y8[ord_idx]; w_sorted = W8[ord_idx]
cum_t = np.cumsum(w_sorted); cum_c = np.cumsum(1 - w_sorted)
cum_y_t = np.cumsum(y_sorted * w_sorted)
cum_y_c = np.cumsum(y_sorted * (1 - w_sorted))
# Avoid zero division
qini = cum_y_t - cum_y_c * np.where(cum_t>0, cum_t, 1) / np.where(cum_c>0, cum_c, 1)
fracs = np.arange(1, len(qini)+1) / len(qini)
# Random baseline is linear from 0 to qini[-1]
rand = np.linspace(0, qini[-1], len(qini))

fig = go.Figure()
fig.add_trace(go.Scatter(x=fracs, y=qini, mode="lines",
                         line=dict(color="#4C8BB8", width=3),
                         name="T-Learner uplift ranking",
                         hovertemplate="Top %{x:.1%}<br>Incremental conv: %{y:.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=fracs, y=rand, mode="lines",
                         line=dict(color="#888", dash="dash"), name="Random"))
auuc = float(np.trapz(qini - rand, fracs))
fig.update_layout(**BASE_LAYOUT,
                  title=f"Qini Curve — AUUC vs random = {auuc:.2f}",
                  xaxis=dict(title="Fraction of population targeted", automargin=True,
                             tickformat=".0%"),
                  yaxis=dict(title="Incremental conversions", automargin=True),
                  height=520,
                  legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
fig.write_html(os.path.join(OUT_DIR, "nb08_qini_curve_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb08_qini_curve_interactive.html")

# Chart 3: Feature importance for differential response via random forest
from sklearn.ensemble import RandomForestClassifier
rf0 = RandomForestClassifier(n_estimators=80, max_depth=8, random_state=42, n_jobs=-1).fit(X8[W8==0], y8[W8==0])
rf1 = RandomForestClassifier(n_estimators=80, max_depth=8, random_state=42, n_jobs=-1).fit(X8[W8==1], y8[W8==1])
diff_imp = rf1.feature_importances_ - rf0.feature_importances_
order = np.argsort(diff_imp)
fig = go.Figure(go.Bar(x=diff_imp[order], y=np.array(fcols)[order], orientation="h",
                       marker_color=["#E74C3C" if v<0 else "#2ECC71" for v in diff_imp[order]],
                       text=[f"{v:+.3f}" for v in diff_imp[order]], textposition="outside",
                       hovertemplate="<b>%{y}</b><br>Δ importance: %{x:+.3f}<extra></extra>"))
fig.add_vline(x=0, line_color="black")
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=160, r=60, t=90, b=80)},
                  title="Feature Importance Difference (Treated − Control) — Drivers of Differential Response",
                  xaxis=dict(title="Treated − Control feature importance", automargin=True),
                  yaxis=dict(automargin=True), height=500)
fig.write_html(os.path.join(OUT_DIR, "nb08_feature_importance_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb08_feature_importance_interactive.html")


  ✓ nb08_qini_curve_interactive.html


  ✓ nb08_feature_importance_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb08/` so it can be dropped straight into the
blog post.


In [19]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# Uplift Modeling — Results-Display Tables (stratified uplift, model comparison)
import os, pandas as pd

# Try loading the csvs saved by earlier cells; recompute a summary if missing
strat_csv = os.path.join(OUT_DIR, "nb08_uplift_by_buyer_type.csv")
summary_csv = os.path.join(OUT_DIR, "nb08_uplift_summary.csv")

if os.path.exists(strat_csv):
    strat = pd.read_csv(strat_csv)
    # Expect columns: buyer_type, n, baseline_conv, treated_conv, observed_lift, predicted_uplift
    cols = [c for c in ["buyer_type","n","baseline_conv","treated_conv","observed_lift","predicted_uplift"]
            if c in strat.columns]
    display_df = strat[cols].copy()
    for c in cols:
        if c in ("baseline_conv","treated_conv","observed_lift","predicted_uplift"):
            display_df[c] = display_df[c].astype(float).apply(lambda v: f"{v*100:+.3f}%" if abs(v)<1 else f"{v:+.3f}")
        elif c == "n":
            display_df[c] = display_df[c].astype(int).apply(lambda v: f"{v:,}")
    header_map = dict(buyer_type="Buyer type", n="n",
                      baseline_conv="Baseline conv.",
                      treated_conv="Treated conv.",
                      observed_lift="Observed lift",
                      predicted_uplift="Predicted τ̂(x)")
    headers = [header_map[c] for c in cols]
    cell_cols = [display_df[c].tolist() for c in cols]
    fig = table_card("Uplift Stratified by Buyer Type — Disentangling Customer Quality from Treatment Effect",
                     headers, cell_cols, [160]*len(cols), height_extra=90)
    fig.write_html(os.path.join(OUT_DIR, "nb08_uplift_stratified_interactive.html"), **PLOTLY_KW)
    fig.show()
else:
    print("  ! nb08_uplift_by_buyer_type.csv not found — run earlier cells first")

if os.path.exists(summary_csv):
    summ = pd.read_csv(summary_csv)
    headers = list(summ.columns)
    cell_cols = [summ[c].astype(str).tolist() for c in headers]
    fig = table_card("Uplift Modeling Summary — T-Learner / S-Learner / Qini",
                     headers, cell_cols, [160]*len(headers), height_extra=90)
    fig.write_html(os.path.join(OUT_DIR, "nb08_uplift_summary_interactive.html"), **PLOTLY_KW)
    fig.show()
else:
    print("  ! nb08_uplift_summary.csv not found — run earlier cells first")
print("  ✓ nb08 uplift result cards saved (where source CSVs existed)")


  ✓ nb08 uplift result cards saved (where source CSVs existed)
